In [1]:
# ===============================
# IMPORTS
# ===============================
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from tqdm import tqdm
import numpy as np
import random
import os
from pathlib import Path
import glob
from torch.utils.data import Dataset, DataLoader

In [2]:
# ===============================
# HELPER FUNCTIONS FROM REPO
# ===============================

INPUT_BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')

OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


seed_everything(42)

In [3]:

# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')


def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)







def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = ["blues", "classical", "country", "disco", "hiphop",
"jazz", "metal", "pop", "reggae", "rock"
]
    target_length = target_sr * duration
    
    # Get noise files from read-only input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        # Create output directories in the writable /kaggle/working/ directory
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Basic Resampling check (if needed)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)



def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    # Find all .wav files in the input directory
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"Warning: No .wav files found in {input_dir}")
        return

    for wav_path in wav_files:
        # Replicate directory structure
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        
        # Ensure the target directory exists in /kaggle/working/
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Process and save
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        
        torch.save(mel_spec_db, out_path)
    
    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")


In [4]:
# ===============================
# GENERATE SYNTHETIC DATASET
# ===============================

generate_synthetic_dataset(
    STEMS_PATH,
    NOISE_PATH,
    OUTPUT_PATH,
    samples_per_genre=50
)

question 1

In [5]:
wav_files = glob.glob('/kaggle/working/synthetic_mashups/train/**/*.wav', recursive=True)
print("q1 ",len(wav_files))

q1  500


In [6]:
# ===============================
# LOAD AUDIO
# ===============================

example_file = wav_files[0]

waveform, sr = torchaudio.load(example_file)

print("q2 ",waveform.shape)
print(sr)

q2  torch.Size([2, 661500])
22050


question 3

In [7]:
# ===============================
# FEATURE EXTRACTION
# ===============================

INPUT_DIR = '/kaggle/working/synthetic_mashups/train'
OUTPUT_DIR = '/kaggle/working/features/train'

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)

Successfully saved 500 feature files to /kaggle/working/features/train


In [8]:
# ===============================
# LOAD FEATURE
# ===============================

pt_files = glob.glob('/kaggle/working/features/train/**/*.pt', recursive=True)

feature = torch.load(pt_files[0])

print(feature.shape)

torch.Size([2, 128, 1292])


question 4

In [9]:
# ===============================
# DATASET
# ===============================

class PrecomputedFeatureDataset(Dataset):

    def __init__(self, features_dir):

        self.files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)

        self.genres = sorted([
            'blues','classical','country','disco','hiphop',
            'jazz','metal','pop','reggae','rock'
        ])

        self.genre_to_idx = {g:i for i,g in enumerate(self.genres)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        file_path = self.files[idx]

        genre = Path(file_path).parent.name
        label = self.genre_to_idx[genre]

        feature = torch.load(file_path)

        return feature, label

In [10]:
# ===============================
# DATA LOADERS
# ===============================

dataset = PrecomputedFeatureDataset('/kaggle/working/features/train')

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset,[train_size,val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [11]:
# ===============================
# CRNN MODEL
# ===============================

class CRNN(nn.Module):

    def __init__(self, num_classes=10):

        super().__init__()

        # CNN BACKBONE
        self.cnn = nn.Sequential(

            nn.Conv2d(1,32,kernel_size=3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # LSTM
        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=64,
            batch_first=True,
            bidirectional=True
        )

        # CLASSIFIER
        self.fc = nn.Linear(128, num_classes)

    def forward(self,x):

        x = self.cnn(x)

        # shape: (batch,64,32,time)
        b,c,f,t = x.shape

        # reshape for LSTM
        x = x.permute(0,3,1,2)
        x = x.reshape(b,t,c*f)

        x,_ = self.lstm(x)

        x,_ = torch.max(x,dim=1)

        logits = self.fc(x)

        return logits

question 5

In [12]:
model = CRNN()

lstm_params = sum(
    p.numel() for p in model.lstm.parameters() if p.requires_grad
)

print(lstm_params)

1082368


question 6

In [13]:
# ===============================
# TRAINING SETUP
# ===============================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CRNN(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

In [14]:
for epoch in range(num_epochs):

    model.train()
    train_loss = 0

    for x,y in train_loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        preds = model(x)

        loss = criterion(preds,y)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    print("Epoch",epoch,"Loss",train_loss/len(train_loader))

RuntimeError: Given groups=1, weight of size [32, 1, 3, 3], expected input[32, 2, 128, 1292] to have 1 channels, but got 2 channels instead